# Análise de Machine Learning - Produção de Energia Marítima

Previsão e classificação da produção de óleo com base em variáveis operacionais
da plataforma (produção de condensado, gás, água e variáveis de injeção).

Modelos aplicados:

1. KNN Regressor
2. Regressão Linear Simples
3. Regressão Linear Múltipla
4. Regressão Logística

Este notebook segue a mesma metodologia e os mesmos parâmetros do script
`analise_ml_producao.py` (base completa, `random_state=42`, divisão 70/30),
de modo que os resultados aqui reproduzidos sejam idênticos aos do relatório
`RELATORIO_ANALISE_MODELOS_ML.md`.
Observação metodológica: as variáveis de produção de gás, água e condensado pertencem ao mesmo registro temporal da produção de óleo. Assim, o notebook deve ser interpretado como exercício supervisionado e análise de associação entre volumes do mesmo período, não como previsão operacional antecipada de meses futuros.


## 1. Carregar e Inspecionar a Base

Importar bibliotecas, carregar o dataset e exibir dimensões e estatísticas iniciais.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')

RANDOM_STATE = 42
TEST_SIZE = 0.30
MAX_SCATTER_POINTS = 3000   # amostragem apenas para os gráficos de dispersão
KNN_CV_SAMPLE_SIZE = 6000   # amostra usada só na validação cruzada do k (custo computacional)

OUTPUT_DIR = Path('resultados_ml')
OUTPUT_DIR.mkdir(exist_ok=True)

# Carregar dados (base completa, sem amostragem)
data_path = Path('producao_maritima_tratada.csv')
df = pd.read_csv(data_path, encoding='utf-8')

print(f"Shape: {df.shape}")
print(f"\nTipos de dados (primeiras colunas):\n{df.dtypes[:10]}")

## 2. Limpar Dados e Remover Colunas Sem Variação

Converter as colunas numéricas para `float`, preencher valores ausentes com zero
(ausência de medição equivale a ausência daquele tipo de volume no registro),
remover duplicatas e descartar colunas sem variação (constantes), pois elas não
ajudam o modelo e podem causar instabilidade numérica em modelos lineares.

In [ ]:
target_col = 'producao_oleo_m3'
feature_cols_originais = [
    'ano',
    'producao_condensado_m3',
    'producao_gas_associado_mm3',
    'producao_gas_nao_associado_mm3',
    'producao_agua_m3',
    'injecao_gas_mm3',
    'injecao_agua_recuperacao_secundaria_m3',
    'injecao_agua_descarte_m3',
    'injecao_gas_carbonico_mm3',
    'injecao_nitrogenio_mm3',
    'injecao_vapor_agua_t',
    'injecao_polimeros_m3',
    'injecao_outros_fluidos_m3',
]

# Converter para float e preencher NaN com 0
for col in [target_col] + feature_cols_originais:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Remover duplicatas
df = df.drop_duplicates().reset_index(drop=True)

# Remover colunas sem variação (constantes)
feature_cols = [c for c in feature_cols_originais if df[c].nunique() > 1]
colunas_removidas = sorted(set(feature_cols_originais) - set(feature_cols))

print(f"Shape após limpeza: {df.shape}")
print(f"Valores nulos após tratamento: {df[[target_col] + feature_cols_originais].isnull().sum().sum()}")
print(f"Colunas removidas por não terem variação: {colunas_removidas}")
print(f"Features finais utilizadas ({len(feature_cols)}): {feature_cols}")

## 3. Análise Exploratória dos Dados

Observar a distribuição da variável alvo e a correlação entre as variáveis
antes de treinar qualquer modelo.

In [ ]:
print(df[target_col].describe(percentiles=[.25, .5, .75, .90, .95, .99]).round(2))

zero = (df[target_col] == 0).sum()
positivo = (df[target_col] > 0).sum()
print(f"\nRegistros com produção zero: {zero} ({zero / len(df):.2%})")
print(f"Registros com produção positiva: {positivo} ({positivo / len(df):.2%})")

# Heatmap de correlação
plt.figure(figsize=(11, 8))
corr = df[[target_col] + feature_cols].corr(numeric_only=True)
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.4)
plt.title('Correlação entre as variáveis')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'heatmap_correlacao.png', dpi=160, bbox_inches='tight')
plt.show()

print("\nCorrelação com a produção de óleo (ordenada por força):")
print(corr[target_col].drop(target_col).sort_values(key=lambda s: s.abs(), ascending=False).round(4))

# Histograma da produção de óleo
plt.figure(figsize=(8, 5))
sns.histplot(df[target_col], bins=40, kde=True)
plt.title('Distribuição da produção de óleo')
plt.xlabel('Produção de óleo (m³)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'histograma_producao_oleo.png', dpi=160, bbox_inches='tight')
plt.show()

print("\nGráficos salvos: resultados_ml/heatmap_correlacao.png, resultados_ml/histograma_producao_oleo.png")

## 4. Selecionar Target e Features

Separar a variável alvo (`producao_oleo_m3`) das variáveis explicativas,
utilizando a base completa (sem amostragem), conforme o relatório.

In [ ]:
y = df[target_col].copy()
X = df[feature_cols].copy()

print(f"Dataset de trabalho: {X.shape}")
print(f"Target ({target_col}): shape={y.shape}, min={y.min():.2f}, max={y.max():.2f}, media={y.mean():.2f}")

## 5. Dividir Conjunto de Treino e Teste

Separar dados em treino (70%) e teste (30%) preservando reprodutibilidade
(`random_state=42`). Esta divisão é usada pelos três modelos de regressão
(KNN, Linear Simples e Linear Múltipla).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

print(f"Treino: X={X_train.shape}, y={y_train.shape}")
print(f"Teste: X={X_test.shape}, y={y_test.shape}")

## 6. Modelo 1: KNN Regressor

O KNN depende da escala das variáveis, por isso é necessário padronizar
(`StandardScaler`) antes de calcular as distâncias entre os registros. Para
evitar vazamento de dados entre os folds da validação cruzada, a padronização
é encapsulada em um `Pipeline` junto com o KNN, de forma que o `scaler` seja
ajustado de novo a cada fold (e, depois, ajustado uma única vez sobre todo o
treino para o modelo final). O melhor valor de `k` é escolhido por validação
cruzada (valores ímpares entre 1 e 15). Em classificação, valores ímpares
ajudam a reduzir empates de votação; aqui, como o KNN é regressivo, eles são
apenas uma grade simples e tradicional de teste. A validação usa uma amostra
do treino para manter o custo computacional viável.


In [ ]:
# Demonstração do efeito do StandardScaler (apenas ilustrativo)
scaler_demo = StandardScaler()
X_train_scaled_demo = scaler_demo.fit_transform(X_train)
X_test_scaled_demo = scaler_demo.transform(X_test)

print(f"X_train_scaled: mean={X_train_scaled_demo.mean():.6f}, std={X_train_scaled_demo.std():.6f}")
print(f"X_test_scaled: mean={X_test_scaled_demo.mean():.6f}, std={X_test_scaled_demo.std():.6f}")

In [ ]:
# Amostra do treino (dados originais, sem padronizar) apenas para a validação cruzada
X_cv = X_train
y_cv = y_train
if len(X_cv) > KNN_CV_SAMPLE_SIZE:
    X_cv = X_cv.sample(KNN_CV_SAMPLE_SIZE, random_state=RANDOM_STATE)
    y_cv = y_train.loc[X_cv.index]

k_values = list(range(1, 16, 2))
cv = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
cv_mse = []

print("Testando valores de k...")
for k in k_values:
    # StandardScaler dentro do Pipeline: é ajustado de novo em cada fold da CV
    modelo_cv = Pipeline([
        ('padronizacao', StandardScaler()),
        ('knn', KNeighborsRegressor(n_neighbors=k)),
    ])
    scores = cross_val_score(modelo_cv, X_cv, y_cv, cv=cv, scoring='neg_mean_squared_error')
    mse_mean = (-scores).mean()
    cv_mse.append(mse_mean)
    print(f"k={k:2d}: MSE CV={mse_mean:.2f}")

best_k = k_values[int(np.argmin(cv_mse))]
print(f"\nMelhor k: {best_k}")

plt.figure(figsize=(7, 5))
plt.plot(k_values, cv_mse, marker='o', linewidth=2)
plt.xlabel('Valor de k')
plt.ylabel('MSE médio na validação cruzada')
plt.title('KNN - Escolha de k')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'knn_mse_por_k.png', dpi=160, bbox_inches='tight')
plt.show()

# Treinar modelo final com o melhor k, em um único Pipeline ajustado sobre todo o treino
knn_final = Pipeline([
    ('padronizacao', StandardScaler()),
    ('knn', KNeighborsRegressor(n_neighbors=best_k)),
])
knn_final.fit(X_train, y_train)
print(f"KNN treinado com k={best_k}")

In [ ]:
y_pred_knn = knn_final.predict(X_test)

knn_mse = mean_squared_error(y_test, y_pred_knn)
knn_mae = mean_absolute_error(y_test, y_pred_knn)
knn_r2 = r2_score(y_test, y_pred_knn)
knn_rmse = np.sqrt(knn_mse)

print("=" * 60)
print(f"KNN (k={best_k}) - Métricas no Conjunto de Teste")
print("=" * 60)
print(f"MSE:  {knn_mse:>15,.2f}")
print(f"RMSE: {knn_rmse:>15,.2f}")
print(f"MAE:  {knn_mae:>15,.2f}")
print(f"R²:   {knn_r2:>15.4f}")

In [ ]:
def amostra_para_grafico(x, y, n=MAX_SCATTER_POINTS, seed=RANDOM_STATE):
    pontos = pd.DataFrame({"x": np.asarray(x), "y": np.asarray(y)})
    if len(pontos) > n:
        pontos = pontos.sample(n, random_state=seed)
    return pontos

# Real vs. Previsto
pontos = amostra_para_grafico(y_test, y_pred_knn)
min_val, max_val = pontos[["x", "y"]].min().min(), pontos[["x", "y"]].max().max()

plt.figure(figsize=(7, 5))
sns.scatterplot(data=pontos, x="x", y="y", alpha=0.45, s=18)
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
plt.xlabel('Valor real')
plt.ylabel('Valor previsto')
plt.title(f'KNN (k={best_k}) - Real vs. Previsto (R²={knn_r2:.4f})')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'knn_real_vs_previsto.png', dpi=160, bbox_inches='tight')
plt.show()

# Resíduos
residuos_knn = np.asarray(y_test) - np.asarray(y_pred_knn)
pontos_res = amostra_para_grafico(y_pred_knn, residuos_knn)
pontos_res.columns = ["previsto", "residuo"]

plt.figure(figsize=(7, 5))
sns.scatterplot(data=pontos_res, x="previsto", y="residuo", alpha=0.45, s=18)
plt.axhline(0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Valor previsto')
plt.ylabel('Resíduo')
plt.title('KNN - Resíduos')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'knn_residuos.png', dpi=160, bbox_inches='tight')
plt.show()

print("Gráficos salvos: resultados_ml/knn_real_vs_previsto.png, resultados_ml/knn_residuos.png")

## 7. Modelo 2: Regressão Linear Simples

Modela a relação entre uma única variável explicativa e a produção de óleo.
A variável escolhida é `producao_gas_associado_mm3`, por ser a mais
correlacionada com a variável alvo (ver heatmap da seção 3).

In [ ]:
SIMPLE_FEATURE = 'producao_gas_associado_mm3'

modelo_simples = LinearRegression()
modelo_simples.fit(X_train[[SIMPLE_FEATURE]], y_train)
y_pred_lr_simples = modelo_simples.predict(X_test[[SIMPLE_FEATURE]])

lrs_mse = mean_squared_error(y_test, y_pred_lr_simples)
lrs_mae = mean_absolute_error(y_test, y_pred_lr_simples)
lrs_r2 = r2_score(y_test, y_pred_lr_simples)
lrs_rmse = np.sqrt(lrs_mse)

print("=" * 60)
print("Regressão Linear Simples - Métricas no Conjunto de Teste")
print("=" * 60)
print(f"Feature usada: {SIMPLE_FEATURE}")
print(f"Coeficiente: {modelo_simples.coef_[0]:.6f}")
print(f"Intercepto:  {modelo_simples.intercept_:.6f}")
print(f"MSE:  {lrs_mse:>15,.2f}")
print(f"RMSE: {lrs_rmse:>15,.2f}")
print(f"MAE:  {lrs_mae:>15,.2f}")
print(f"R²:   {lrs_r2:>15.4f}")

In [ ]:
pontos = pd.DataFrame({SIMPLE_FEATURE: X_test[SIMPLE_FEATURE], target_col: y_test, "previsto": y_pred_lr_simples})
if len(pontos) > MAX_SCATTER_POINTS:
    pontos = pontos.sample(MAX_SCATTER_POINTS, random_state=RANDOM_STATE)
ordem = np.argsort(pontos[SIMPLE_FEATURE].to_numpy())

plt.figure(figsize=(7, 5))
sns.scatterplot(data=pontos, x=SIMPLE_FEATURE, y=target_col, alpha=0.35, s=18)
plt.plot(pontos[SIMPLE_FEATURE].to_numpy()[ordem], pontos["previsto"].to_numpy()[ordem], color='red', linewidth=2)
plt.title('Regressão Linear Simples - Reta Ajustada')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'regressao_simples_reta.png', dpi=160, bbox_inches='tight')
plt.show()

residuos_simples = np.asarray(y_test) - np.asarray(y_pred_lr_simples)
pontos_res = amostra_para_grafico(y_pred_lr_simples, residuos_simples)
pontos_res.columns = ["previsto", "residuo"]

plt.figure(figsize=(7, 5))
sns.scatterplot(data=pontos_res, x="previsto", y="residuo", alpha=0.45, s=18)
plt.axhline(0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Valor previsto')
plt.ylabel('Resíduo')
plt.title('Regressão Linear Simples - Resíduos')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'regressao_simples_residuos.png', dpi=160, bbox_inches='tight')
plt.show()

print("Gráficos salvos: resultados_ml/regressao_simples_reta.png, resultados_ml/regressao_simples_residuos.png")

## 8. Modelo 3: Regressão Linear Múltipla

Utiliza todas as variáveis explicativas simultaneamente. Além das métricas
de erro, calcula-se o R² ajustado e, via `statsmodels`, os coeficientes,
erros padrão e p-valores de cada variável. Também é calculado o VIF para
avaliar multicolinearidade entre as variáveis explicativas.


In [ ]:
modelo_multiplo = LinearRegression()
modelo_multiplo.fit(X_train, y_train)
y_pred_lr_multipla = modelo_multiplo.predict(X_test)

lrm_mse = mean_squared_error(y_test, y_pred_lr_multipla)
lrm_mae = mean_absolute_error(y_test, y_pred_lr_multipla)
lrm_r2 = r2_score(y_test, y_pred_lr_multipla)
lrm_rmse = np.sqrt(lrm_mse)

n_amostras, n_features = X_test.shape
lrm_r2_ajustado = 1 - (1 - lrm_r2) * (n_amostras - 1) / (n_amostras - n_features - 1)

print("=" * 60)
print("Regressão Linear Múltipla - Métricas no Conjunto de Teste")
print("=" * 60)
print(f"MSE:         {lrm_mse:>15,.2f}")
print(f"RMSE:        {lrm_rmse:>15,.2f}")
print(f"MAE:         {lrm_mae:>15,.2f}")
print(f"R²:          {lrm_r2:>15.4f}")
print(f"R² ajustado: {lrm_r2_ajustado:>15.4f}")

In [ ]:
X_train_ols = sm.add_constant(X_train, has_constant='add')
ols = sm.OLS(y_train, X_train_ols).fit()

coeficientes = pd.DataFrame({
    "feature": ols.params.index,
    "coeficiente": ols.params.values,
    "erro_padrao": ols.bse.values,
    "p_valor": ols.pvalues.values,
}).sort_values("coeficiente", key=lambda s: s.abs(), ascending=False)

coeficientes.to_csv(OUTPUT_DIR / "coeficientes_regressao_multipla.csv", index=False)
print("Principais coeficientes (ordenados por magnitude):")
print(coeficientes.to_string(index=False))

vif_dados = pd.DataFrame({
    "feature": X_train_ols.columns,
    "VIF": [
        variance_inflation_factor(X_train_ols.values, i)
        for i in range(X_train_ols.shape[1])
    ],
}).sort_values("VIF", ascending=False).reset_index(drop=True)

vif_dados.to_csv(OUTPUT_DIR / "vif_regressao_multipla.csv", index=False)
print("\nFator de Inflação da Variância (VIF):")
print(vif_dados.to_string(index=False))


In [ ]:
pontos = amostra_para_grafico(y_test, y_pred_lr_multipla)
min_val, max_val = pontos[["x", "y"]].min().min(), pontos[["x", "y"]].max().max()

plt.figure(figsize=(7, 5))
sns.scatterplot(data=pontos, x="x", y="y", alpha=0.45, s=18, color='green')
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
plt.xlabel('Valor real')
plt.ylabel('Valor previsto')
plt.title(f'Regressão Linear Múltipla - Real vs. Previsto (R²={lrm_r2:.4f})')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'regressao_multipla_real_vs_previsto.png', dpi=160, bbox_inches='tight')
plt.show()

residuos_multipla = np.asarray(y_test) - np.asarray(y_pred_lr_multipla)
pontos_res = amostra_para_grafico(y_pred_lr_multipla, residuos_multipla)
pontos_res.columns = ["previsto", "residuo"]

plt.figure(figsize=(7, 5))
sns.scatterplot(data=pontos_res, x="previsto", y="residuo", alpha=0.45, s=18, color='green')
plt.axhline(0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Valor previsto')
plt.ylabel('Resíduo')
plt.title('Regressão Linear Múltipla - Resíduos')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'regressao_multipla_residuos.png', dpi=160, bbox_inches='tight')
plt.show()

print("Gráficos salvos: resultados_ml/regressao_multipla_real_vs_previsto.png, resultados_ml/regressao_multipla_residuos.png")

## 9. Modelo 4: Regressão Logística

Problema de classificação binária: `produziu_oleo = 1` quando
`producao_oleo_m3 > 0` e `0` quando é igual a zero. Como o alvo muda, é feita
uma nova divisão treino/teste (estratificada) sobre as mesmas features.
`StandardScaler` é aplicado para melhorar a estabilidade da otimização e para
que a regularização padrão da regressão logística trate variáveis em escalas
diferentes de forma mais equilibrada.


In [ ]:
df_log = df.copy()
df_log['produziu_oleo'] = (df_log[target_col] > 0).astype(int)

X_log = df_log[feature_cols]
y_log = df_log['produziu_oleo']

X_log_train, X_log_test, y_log_train, y_log_test = train_test_split(
    X_log, y_log,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_log,
)

modelo_logistico = Pipeline([
    ('padronizacao', StandardScaler()),
    ('logistica', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
modelo_logistico.fit(X_log_train, y_log_train)

y_pred_log = modelo_logistico.predict(X_log_test)
y_proba_log = modelo_logistico.predict_proba(X_log_test)[:, 1]

log_metricas = {
    "Acuracia": accuracy_score(y_log_test, y_pred_log),
    "Precisao": precision_score(y_log_test, y_pred_log, zero_division=0),
    "Recall": recall_score(y_log_test, y_pred_log, zero_division=0),
    "F1": f1_score(y_log_test, y_pred_log, zero_division=0),
    "AUC": roc_auc_score(y_log_test, y_proba_log),
}

print("Distribuição das classes na base completa:", y_log.value_counts().to_dict())
print()
for nome, valor in log_metricas.items():
    print(f"{nome}: {valor:.4f}")
print("\nRelatório de classificação:")
print(classification_report(y_log_test, y_pred_log, zero_division=0))

In [ ]:
cm = confusion_matrix(y_log_test, y_pred_log)
disp = ConfusionMatrixDisplay(cm, display_labels=['Não produziu', 'Produziu'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Regressão Logística - Matriz de Confusão')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'logistica_matriz_confusao.png', dpi=160, bbox_inches='tight')
plt.show()

fpr, tpr, _ = roc_curve(y_log_test, y_proba_log)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"AUC = {log_metricas['AUC']:.4f}", linewidth=2)
plt.plot([0, 1], [0, 1], 'r--')
plt.xlabel('Taxa de falso positivo')
plt.ylabel('Taxa de verdadeiro positivo')
plt.title('Regressão Logística - Curva ROC')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'logistica_roc.png', dpi=160, bbox_inches='tight')
plt.show()

print("Gráficos salvos: resultados_ml/logistica_matriz_confusao.png, resultados_ml/logistica_roc.png")

## 10. Comparação Final dos Modelos

Consolidar as métricas dos três modelos de regressão e do modelo de
classificação em tabelas resumo, salvas em `resultados_ml/`.

In [ ]:
resumo_regressao = pd.DataFrame([
    {"Modelo": "KNN Regressor", "MSE": knn_mse, "RMSE": knn_rmse, "MAE": knn_mae, "R2": knn_r2, "R2_ajustado": np.nan},
    {"Modelo": "Regressao Linear Simples", "MSE": lrs_mse, "RMSE": lrs_rmse, "MAE": lrs_mae, "R2": lrs_r2, "R2_ajustado": np.nan},
    {"Modelo": "Regressao Linear Multipla", "MSE": lrm_mse, "RMSE": lrm_rmse, "MAE": lrm_mae, "R2": lrm_r2, "R2_ajustado": lrm_r2_ajustado},
])
resumo_classificacao = pd.DataFrame([{"Modelo": "Regressao Logistica", **log_metricas}])

resumo_regressao.to_csv(OUTPUT_DIR / "resumo_modelos_regressao.csv", index=False)
resumo_classificacao.to_csv(OUTPUT_DIR / "resumo_modelo_logistico.csv", index=False)

print("=" * 70)
print("RESUMO - MODELOS DE REGRESSAO")
print("=" * 70)
print(resumo_regressao.to_string(index=False))

print("\n" + "=" * 70)
print("RESUMO - MODELO DE CLASSIFICACAO")
print("=" * 70)
print(resumo_classificacao.to_string(index=False))

print(f"\nGráficos e tabelas salvos em: {OUTPUT_DIR.resolve()}")
print("Tabela adicional de diagnóstico: resultados_ml/vif_regressao_multipla.csv")
